In [ ]:
!huggingface-cli login

## Library

In [1]:
!pip install bitsandbytes accelerate

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
!git clone https://github.com/openai/human-eval
!pip install -e human-eval

Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///ocean/projects/cis250075p/rjoshi4/human-eval
  Preparing metadata (setup.py) ... done
  Attempting uninstall: human-eval
    Found existing installation: human-eval 1.0
    Uninstalling human-eval-1.0:
      Successfully uninstalled human-eval-1.0
  Running setup.py develop for human-eval


In [3]:
!pip install transformers
!pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [4]:
cd human-eval

/ocean/projects/cis250075p/rjoshi4/human-eval


## 1. Self-Refine prompts

In [ ]:
def build_self_refine_input(tokenizer,model, completion1: str, max_length=600) -> str:
    # Feedback Generation
    feedback_prompt = (
        "Please provide feedback on the Python function below.\n"
        "Point out any errors in logic, calculations, or missing steps.\n"
        "Be specific and constructive.\n\n"
        + completion1 +
        "\nFeedback:\n"
    )
    input_feedback = tokenizer(feedback_prompt, return_tensors="pt").to(model.device)
    feedback_output = model.generate(
        input_feedback.input_ids,
        max_length=max_length,
        repetition_penalty=1.5,
        temperature=0.5,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    feedback_gen = tokenizer.decode(feedback_output[0], skip_special_tokens=True)
    feedback = feedback_gen[len(feedback_prompt):] # remove feedback_prompt
    prompt = (
        "Here is the original function:\n\n"
        + completion1.strip() +
        "\n\nFeedback on the function:\n"
        + feedback.strip() +
        "\n\nPlease revise the code to fix any issues based on the feedback above. "
        "Only output the final corrected Python function. Do not include markdown formatting like ```python or explanations. Just raw Python code." +
        "\nRefined Code:\n"
    )
    return prompt 

In [ ]:
import re
def extract_function_only(text: str) -> str:
    # Extract first code block if present
    code_block_match = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    if code_block_match:
        return code_block_match.group(1).strip()
    
    # Fallback: return text if no backticks
    return text.strip()

In [ ]:
from human_eval.data import read_problems, write_jsonl
import itertools
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

problems = read_problems()

model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# Apply 4-bit quantization
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #quantization_config=bnb_config,
    device_map="auto"
)

samples_first = []
samples_second = []
for problem_id in tqdm(problems.keys()):
    #  Prompt 1
    coding_prompt = problems[problem_id]["prompt"]
    prompt1_header = "You are an expert Python programmer, and here is your task: Complete the following python function: \n"
    prompt1 = prompt1_header + coding_prompt
    input1 = tokenizer(prompt1, return_tensors="pt").to(model.device)

    output1 = model.generate(
        input1.input_ids,
        max_length=500,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_code1 = tokenizer.decode(output1[0], skip_special_tokens=True)

    completion1 = generated_code1[len(prompt1):] # remove header and coding prompt

    # Prompt 2 refine_prompt
    prompt2 = build_self_refine_input(tokenizer,model, coding_prompt + completion1) # original generated code + feedback on that

    input2 = tokenizer(prompt2, return_tensors="pt").to(model.device)

    output2 = model.generate(
        input2.input_ids,
        max_length=1000,
        repetition_penalty=1.1,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    generated_code2 = tokenizer.decode(output2[0], skip_special_tokens=True).strip()

    completion2 = generated_code2[len(prompt2):] # Self-Refine
    completion2 = extract_function_only(completion2)

    samples_first.append({
        "task_id": problem_id,
        "completion": completion1
    })

    samples_second.append({
        "task_id": problem_id,
        "completion": completion2
    })
    # print(f"Problem ID: {problem_id}")
    # print("=" * 40)
    # print("Prompt 1:")
    # print(prompt1)
    # print("=" * 40)
    # print("Prompt + Completion (First):")
    # print(generated_code1)
    # print("=" * 40)
    # print("Just Completion (First):")
    # print(completion1)
    # print("\n")
    # print("Prompt 2:")
    # print(prompt2)
    # print("=" * 40)
    # print("Prompt + Completion (Second):")
    # print(generated_code2)
    # print("=" * 40)
    # print("Just Completion (Second):")
    # print(completion2)
    # print("\n")
    # break
write_jsonl("humaneval_samples1.jsonl", samples_first)
write_jsonl("humaneval_samples2.jsonl", samples_second)

/jet/home/rjoshi4/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/packages/AI/pytorch_23.02-1.13.1-py3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 164/164 [16:15<00:00,  5.95s/it]


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()
!ls

data					humaneval_samples3.jsonl_results.jsonl
human_eval				humaneval_samples4.jsonl
human_eval.egg-info			humaneval_samples4.jsonl_results.jsonl
humaneval_samples1.jsonl		LICENSE
humaneval_samples1.jsonl_results.jsonl	README.md
humaneval_samples2.jsonl		requirements.txt
humaneval_samples2.jsonl_results.jsonl	setup.py
humaneval_samples3.jsonl


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [11]:
!python human_eval/evaluate_functional_correctness.py humaneval_samples1.jsonl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading samples...
164it [00:00, 42820.33it/s]
Running test suites...
100%|████████████████████████████████████████| 164/164 [00:00<00:00, 206.02it/s]
Writing results to humaneval_samples1.jsonl_results.jsonl...
100%|██████████████████████████████████████| 164/164 [00:00<00:00, 61930.84it/s]
{'pass@1': 0.25}


In [12]:
!python human_eval/evaluate_functional_correctness.py humaneval_samples2.jsonl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading samples...
164it [00:00, 40505.59it/s]
Running test suites...
100%|████████████████████████████████████████| 164/164 [00:00<00:00, 196.78it/s]
Writing results to humaneval_samples2.jsonl_results.jsonl...
100%|███████████████████████████████████████| 164/164 [00:00<00:00, 5745.67it/s]
{'pass@1': 0.14634146341463414}


In [13]:
import json

file_path = "humaneval_samples1.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict1 = {i: item for i, item in enumerate(data)}

file_path = "humaneval_samples2.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict2 = {i: item for i, item in enumerate(data)}

In [ ]:
# Get Results
count_i_c = 0
count_c_i = 0
correct_1 = 0
correct_2 = 0

num_problems = len(problems)
for i in range(len(data_dict1)):
    if data_dict1[i]['passed']:
        correct_1 += 1
    if data_dict2[i]['passed']:
        correct_2 += 1
    if data_dict1[i]['passed'] and not data_dict2[i]['passed']:
        count_c_i += 1
    elif not data_dict1[i]['passed'] and data_dict2[i]['passed']:
        count_i_c += 1

print("Accuracy@t1: " + str(correct_1 / num_problems))
print("Accuracy@t2: " + str(correct_2 / num_problems))
print("delta(t1,t2): " + str((correct_2 - correct_1) / num_problems))
print("delta(t1,t2) i to c: " + str(count_i_c / num_problems))
print("delta(t2,t1) c to i: " + str(count_c_i / num_problems))

Accuracy@t1: 0.25
Accuracy@t2: 0.14634146341463414
delta(t1,t2): -0.10365853658536585
delta(t1,t2) i to c: 0.018292682926829267
delta(t2,t1) c to i: 0.12195121951219512


## Chain-of-Thought in Neural Code Generation: From and For Lightweight Language Models

In [ ]:
!git clone https://github.com/NTDXYG/COTTON.git

In [ ]:
!pip install sentencepiece

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import os
os.chdir('/ocean/projects/cis250075p/user/COTTON')

In [ ]:
from nlp2 import set_seed
from LLAMA_Model import LLAMASeq2Seq
set_seed(42)
import itertools
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "codellama/CodeLlama-7b-Python-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

guidance_model = LLAMASeq2Seq(
    base_model_path="codellama/CodeLlama-7b-Python-hf",
    add_eos_token=False,
    adapter="lora",
    load_adapter_path="save_model/checkpoint-best-bleu",
    source_len=256,
    cutoff_len=512
)


/jet/home/rjoshi4/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/packages/AI/pytorch_23.02-1.13.1-py3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


***** CUDA.empty_cache() *****


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:07<00:00,  3.51s/it]


## You may need to clear cache storage

In [ ]:
def build_cotton(question: str) -> str:
    """
    Constructs the full prompt for the refinement step in Self-Refine prompting.
    
    Args:
        tokenizer
        question
        
    Returns:
        str: The full prompt to be passed to the model for generating the refined code.
    """
    # Instruction Generation
    cot = guidance_model.predict(question)
    
    guidance_prompt = (
        "Please follow below instructions to code the Python function.\n"
        + cot
    )
    return guidance_prompt

In [ ]:
from human_eval.data import read_problems, write_jsonl
import itertools
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

problems = read_problems()

model_name = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    #quantization_config=bnb_config,
    device_map="auto"
)


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.76s/it]


In [ ]:

samples_first = []
samples_second = []
samples_third = []
samples_fourth = []
for problem_id in tqdm(problems.keys()):
    #  Prompt 1
    coding_prompt = problems[problem_id]["prompt"]
    prompt1_header = "You are an expert Python programmer, and here is your task: Complete the following python function: \n"
    prompt1 = prompt1_header + coding_prompt
    input1 = tokenizer(prompt1, return_tensors="pt").to(model.device)

    output1 = model.generate(
        input1.input_ids,
        max_length=500,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_code1 = tokenizer.decode(output1[0], skip_special_tokens=True)

    completion1 = generated_code1[len(prompt1):] # remove header and coding prompt

    # Prompt 2 refine_prompt
    prompt2_header = build_cotton(coding_prompt)
    prompt2 = prompt2_header + generated_code1[len(prompt1_header):] # adds just the coding portion of prompt 1's response

    input2 = tokenizer(prompt2, return_tensors="pt").to(model.device)

    output2 = model.generate(
        input2.input_ids,
        max_length=1000,
        repetition_penalty=1.1,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    generated_code2 = tokenizer.decode(output2[0], skip_special_tokens=True).strip()

    completion2 = generated_code2[len(prompt2_header) + len(coding_prompt):] # remove header and coding prompt

    samples_first.append({
        "task_id": problem_id,
        "completion": completion1
    })

    samples_second.append({
        "task_id": problem_id,
        "completion": completion2,
    })

    # print(f"Problem ID: {problem_id}")
    # print("=" * 40)
    # print("Prompt 1:")
    # print(prompt1)
    # print("=" * 40)
    # print("Prompt + Completion (First):")
    # print(generated_code1)
    # print("=" * 40)
    # print("Just Completion (First):")
    # print(completion1)
    # print("\n")
    # print("Prompt 2:")
    # print(prompt2)
    # print("=" * 40)
    # print("Prompt + Completion (Second):")
    # print(generated_code2)
    # print("=" * 40)
    # print("Just Completion (Second):")
    # print(completion2)
    # print("\n")
    # break
write_jsonl("humaneval_samples1.jsonl", samples_first)
write_jsonl("humaneval_samples2.jsonl", samples_second)

 88%|███████████████████████████████████████████████████████████████████████████████████████████████▋             | 144/164 [16:42<02:48,  8.40s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

390

In [ ]:
%cd /ocean/projects/cis250075p/rjoshi4/human-eval
!ls

/ocean/projects/cis250075p/rjoshi4/human-eval
data					humaneval_samples2.jsonl_results.jsonl
human_eval				LICENSE
human_eval.egg-info			README.md
humaneval_samples1.jsonl		requirements.txt
humaneval_samples1.jsonl_results.jsonl	setup.py
humaneval_samples2.jsonl


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
# !python human_eval/evaluate_functional_correctness.py humaneval_samples1.jsonl
# !python human_eval/evaluate_functional_correctness.py humaneval_samples2.jsonl
!evaluate_functional_correctness humaneval_samples1.jsonl
!evaluate_functional_correctness humaneval_samples2.jsonl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading samples...
164it [00:00, 36479.95it/s]
Running test suites...
100%|█████████████████████████████████████████| 164/164 [00:03<00:00, 51.26it/s]
Writing results to humaneval_samples1.jsonl_results.jsonl...
100%|██████████████████████████████████████| 164/164 [00:00<00:00, 58541.77it/s]
{'pass@1': 0.23170731707317074}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Reading samples...
164it [00:00, 37782.37it/s]
Running test suites...
100%|█████████████████████████████████████████| 164/164 [00:03<00:00, 51.17it/s]
Writing results to humaneval_samples2.jsonl_results.jsonl...
100%|███████████████████████████████████████| 164/164 [00:00<00:00, 6422.95it/s]
{'pass@1': 0.23780487804878048}


In [ ]:
import json

file_path = "humaneval_samples1.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict1 = {i: item for i, item in enumerate(data)}

file_path = "humaneval_samples2.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict2 = {i: item for i, item in enumerate(data)}

In [ ]:
# Get Results
count_i_c = 0
count_c_i = 0
correct_1 = 0
correct_2 = 0

num_problems = len(problems)
for i in range(len(data_dict1)):
    if data_dict1[i]['passed']:
        correct_1 += 1
    if data_dict2[i]['passed']:
        correct_2 += 1
    if data_dict1[i]['passed'] and not data_dict2[i]['passed']:
        count_c_i += 1
    elif not data_dict1[i]['passed'] and data_dict2[i]['passed']:
        count_i_c += 1

print("Accuracy@t1: " + str(correct_1 / num_problems))
print("Accuracy@t2: " + str(correct_2 / num_problems))
print("delta(t1,t2): " + str((correct_2 - correct_1) / num_problems))
print("delta(t1,t2) i to c: " + str(count_i_c / num_problems))
print("delta(t2,t1) c to i: " + str(count_c_i / num_problems))

Accuracy@t1: 0.23170731707317074
Accuracy@t2: 0.23780487804878048
delta(t1,t2): 0.006097560975609756
delta(t1,t2) i to c: 0.006097560975609756
delta(t2,t1) c to i: 0.0
